# Ideal Donnan Sorption Model

Predicts how much salt partitions from an external solution into a charged membrane's
water phase, assuming **ideal** (no activity-coefficient) Donnan equilibrium.

Python port of `Donnan_Ideal.m` from the original MATLAB Donnan-Manning Sorption
Analysis code. See `sorption_models.py` in this folder for the underlying math.

**Units:** Css and Csm,w in mol salt / kg water (molality); CAm,w in mol[fixed charge] /
kg water; phiw as a unitless volume fraction (0-1).

In [6]:
import sys, os
sys.path.insert(0, os.getcwd())

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

from sorption_models import run_ideal_donnan
from gui_helpers import EditableTable, labeled

## 1. Membrane parameters

- **zg, zc, zA**: valences of the fixed charge group, counter-ion, and co-ion (e.g. for a
  cation-exchange membrane in NaCl, zg = -1, zc = +1, zA = -1... adjust signs/values to
  match your system; see the original spreadsheet for examples).
- **Phiw (DI)**: membrane water volume fraction equilibrated with DI water.
- **CAm,w (DI)**: fixed-charge concentration (mol/kg water) equilibrated with DI water.

In [7]:
membrane_name = widgets.Text(value="", placeholder="e.g. C2VI2 Photo")
zg = widgets.FloatText(value=-1)
zc = widgets.FloatText(value=1)
zA = widgets.FloatText(value=1)
phiw_DI = widgets.FloatText(value=0.5)
CAmw_DI = widgets.FloatText(value=5.0)

scalar_form = widgets.VBox([
    labeled(membrane_name, "Membrane name (optional)"),
    labeled(zg, "zg (fixed charge valence)"),
    labeled(zc, "zc (counter-ion valence)"),
    labeled(zA, "zA (co-ion valence)"),
    labeled(phiw_DI, "Phiw (DI) (-)"),
    labeled(CAmw_DI, "CAm,w (DI) (m)"),
])
display(scalar_form)

## 2. Concentration-dependent data

One row per external salt concentration you measured/want to predict at.

- **Css (m)** — required: external solution salt molality.
- **phiw_s (-)** — optional: membrane water fraction equilibrated with that salt solution.
  Leave the whole column blank to reuse Phiw (DI) for every point.
- **Csmw measured (m)** — optional: your measured sorbed-salt data. Provide it (for every
  row) to also see the model's RMS log error against your data.

In [3]:
table = EditableTable(["Css (m)", "phiw_s (-) [optional]", "Csmw measured (m) [optional]"], n_rows=3)
display(table.widget)

## 3. Compute

In [9]:
compute_btn = widgets.Button(description="Compute", button_style="primary")
out = widgets.Output()

def on_compute(_btn):
    with out:
        clear_output(wait=True)
        try:
            Css = table.get_column(0)
            if not Css:
                print("Enter at least one row with a Css value.")
                return
            phiw_s = table.get_column(1)
            Csmw_meas = table.get_column(2)

            result, rmsle_val = run_ideal_donnan(
                zg.value, zc.value, zA.value, phiw_DI.value, CAmw_DI.value,
                Css, phiw_s or None, Csmw_meas or None,
            )

            title = membrane_name.value or "Ideal Donnan model"
            print(f"=== {title} ===")
            display(result)
            if rmsle_val is not None:
                print(f"RMSLE (vs. measured data): {rmsle_val:.4g}")

            fig, ax = plt.subplots(figsize=(5, 4))
            ax.plot(result["Css (m)"], result["Csm,w Ideal Donnan (m)"], "o-", label="Ideal Donnan (predicted)")
            if "Csm,w measured (m)" in result.columns:
                ax.plot(result["Css (m)"], result["Csm,w measured (m)"], "s", label="Measured")
            ax.set_xlabel("Css (m)")
            ax.set_ylabel("Csm,w (m)")
            ax.set_title(title)
            ax.set_xscale('log')
            ax.set_yscale('log')
            ax.legend()
            fig.tight_layout()
            plt.show()
        except ValueError as e:
            print(f"Input error: {e}")

compute_btn.on_click(on_compute)
display(widgets.VBox([compute_btn, out]))